# Run a trained model on new drawings (inference)
Puts the model's answer on new beam PDFs (no annotation needed) into review pages you can open and correct.
Runtime > Change runtime type > **T4 GPU** (CPU also works, slower). Run the cells top to bottom (Shift+Enter).

**1. Connect Google Drive**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

**2. Get the latest code from GitHub** (uses your GH_TOKEN secret)

In [ ]:
import os, subprocess, sys
from google.colab import userdata
token = userdata.get("GH_TOKEN")
if os.path.exists("/content/GNN-BeamRebarDetector"):
    subprocess.run(["git", "-C", "/content/GNN-BeamRebarDetector", "pull", "-q"])
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://{token}@github.com/yeongit5985/GNN-BeamRebarDetector.git", "/content/GNN-BeamRebarDetector"])
sys.path.insert(0, "/content/GNN-BeamRebarDetector")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdfium2", "rapidocr_onnxruntime"])
print("code ready")

**3. Your folders and the model**: put the new PDFs in `Inference/<RUN>/PDF`. `RUN` is the training run whose model you use
(`gnn_training/<RUN>/model/beam_gnn.pt`). The review pages go to `Inference/<RUN>/<date_time>`.

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone, timedelta
import gnn

BASE = Path("/content/drive/MyDrive/Deep Learning/GNN-BeamRebarDetector")      # your working directory
RUN  = "261021_2212"                                                          # the training run whose model you use
MODEL_PATH = BASE / "gnn_training" / RUN / "model" / "beam_gnn.pt"
PDFS = BASE / "Inference" / RUN / "PDF"                                       # the new drawings

my_tz = timezone(timedelta(hours=8))    # Set Malaysia timezone (UTC+8)
OUT = BASE / "Inference" / "Model_" / RUN / datetime.now(my_tz).strftime("%y%m%d_%H%M")  # graphs and review pages go here
OUT.mkdir(parents=True, exist_ok=True)

model, ck = gnn.load_model(MODEL_PATH)
model = model.to(gnn.default_device())
print(f"model of {RUN} (saved {ck.get('saved')})")

**4. Read each page and run the model** (about a minute per page). Every page of every PDF becomes a graph (built from
the drawing alone, no labels) and the model labels and links it. A page that fails is skipped with the reason.

In [ ]:
import graph_builder as gb
from pdf_backend import pymupdf

img_dir = OUT / "img"
img_dir.mkdir(parents=True, exist_ok=True)

# listed here, not in step 3, so PDFs you add to the folder after step 3 are found
pdf_files = sorted(p for p in PDFS.glob("*") if p.suffix.lower() == ".pdf")
if not pdf_files:
    raise FileNotFoundError(f"no PDF in {PDFS}. Put the drawings there and run this cell again. Just copied them? "
                            "Colab may not see them yet: run drive.mount('/content/drive', force_remount=True) first")
print(f"{len(pdf_files)} PDFs in {PDFS}")

ai_tasks = []
for pdf in pdf_files:
    for page_no in range(len(pymupdf.open(pdf))):
        # a task like the annotated ones: the image is named <pdf>__p001.png, so the review page finds the PDF page
        task = dict(id=len(ai_tasks) + 1, data=dict(image=f"{pdf.stem}__p{page_no + 1:03d}.png"))
        try:
            g = gb.build_graph(pdf, page_no, task, unlabelled=True)
        except Exception as e:
            print(f"{pdf.name} p{page_no + 1}: skipped ({type(e).__name__}: {e})")
            continue
        gb.save_graph(g, OUT / "graphs")
        g = gnn.graph_from_arrays(g["arrays"], g["meta"], f"{pdf.stem}_p{page_no}")
        if any(ck.get(k) and g[k] != ck[k] for k in ("feature_names", "class_names", "pair_kinds")):
            raise ValueError(f"the graphs of this code differ from the ones {RUN} was trained on: train a new model, "
                             "or use the code of that run")
        page = pymupdf.open(pdf)[page_no]
        if getattr(page, "rotation", 0):
            page.remove_rotation()
        pix = page.get_pixmap(dpi=150, colorspace=pymupdf.csRGB)
        pix.save(img_dir / task["data"]["image"])          # the page image the review page shows
        ai_tasks.append(gnn.prediction_task(model, g, task, page.rect.width, page.rect.height, pix.width, pix.height))
        print(f"{pdf.name} p{page_no + 1}: {len(g['meta']['nodes'])} items read")

if not ai_tasks:
    raise RuntimeError("no page could be read: see the reasons above")
(OUT / "ai_answers.json").write_text(json.dumps(ai_tasks), encoding="utf-8")
print(f"{len(ai_tasks)} pages done")

**5. Review pages** of the model's answer (open `review_index.html` in the output folder). Correct them there and Save
JSON to use a page for training. Note: the model's bar pieces are not joined into whole bars yet, so a long bar can
show as several pieces and a label links to one of them.

In [ ]:
r = subprocess.run([sys.executable, "/content/GNN-BeamRebarDetector/review_relations.py", str(OUT / "ai_answers.json"),
                    "--images", str(img_dir), "--pdf-dir", str(PDFS), "--out", str(OUT)], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
print(f"open {OUT / 'review_index.html'}")